<a href="https://colab.research.google.com/github/gayatrishekharcs/BIO_INSPIRED_LAB/blob/main/Ant_Colony_Optimization_(ACO).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import random

# -------------------------------
# 1. HOSPITAL RESOURCE DETAILS
# -------------------------------

beds = {
    "General": 3,
    "ICU": 2
}

equipment = {
    "Monitor": 3,
    "Ventilator": 2
}

total_staff = 5

# 5 = Highest priority, 1 = Lowest priority
patients = [
    {"id": 1, "priority": 5, "wait": 4,
     "staff": 2, "beds": ["ICU"],
     "equipment": ["Ventilator", "Monitor"]},

    {"id": 2, "priority": 4, "wait": 3,
     "staff": 1, "beds": ["General", "ICU"],
     "equipment": ["Monitor"]},

    {"id": 3, "priority": 3, "wait": 2,
     "staff": 1, "beds": ["General"],
     "equipment": ["Monitor"]},

    {"id": 4, "priority": 5, "wait": 5,
     "staff": 2, "beds": ["ICU"],
     "equipment": ["Ventilator"]},

    {"id": 5, "priority": 2, "wait": 1,
     "staff": 1, "beds": ["General"],
     "equipment": ["Monitor"]},

    {"id": 6, "priority": 3, "wait": 2,
     "staff": 1, "beds": ["General", "ICU"],
     "equipment": ["Monitor"]}
]

# Resource allocation costs
bed_cost = {"General": 5, "ICU": 15}
equipment_cost = {"Monitor": 4, "Ventilator": 10}

# -------------------------------
# 2. ACO PARAMETERS
# -------------------------------

ants = 20
iterations = 50
alpha = 1
beta = 2
evaporation = 0.5
Q = 100

# Generate possible resource combinations
options = {}

for p in patients:
    options[p["id"]] = []

    for b in p["beds"]:
        for e in p["equipment"]:
            options[p["id"]].append((b, e))

# Initialize pheromones
pheromone = {}

for p in patients:
    for option in options[p["id"]]:
        pheromone[(p["id"], option)] = 1.0

    pheromone[(p["id"], "WAIT")] = 1.0


# -------------------------------
# 3. CONSTRUCT A SOLUTION
# -------------------------------

def construct_solution():

    remaining_beds = beds.copy()
    remaining_equipment = equipment.copy()
    available_staff = total_staff

    allocation = {}
    total_cost = 0

    # Give preference to high-priority patients
    ordered_patients = sorted(
        patients,
        key=lambda p: (-p["priority"], -p["wait"])
    )

    for p in ordered_patients:

        pid = p["id"]
        choices = []

        # Find feasible resource combinations
        for option in options[pid]:

            b, e = option

            if (remaining_beds[b] > 0 and
                remaining_equipment[e] > 0 and
                available_staff >= p["staff"]):

                cost = (
                    bed_cost[b] +
                    equipment_cost[e] +
                    p["staff"] * 3
                )

                choices.append((option, cost))

        # Include waiting as an option
        wait_cost = 100 + p["priority"] * 50 + p["wait"] * 5

        choices.append(("WAIT", wait_cost))

        # Calculate selection probabilities
        weights = []

        for option, cost in choices:

            tau = pheromone[(pid, option)]
            eta = 1 / (1 + cost)

            weight = (tau ** alpha) * (eta ** beta)

            weights.append(weight)

        total_weight = sum(weights)

        probabilities = [
            w / total_weight for w in weights
        ]

        selected_index = random.choices(
            range(len(choices)),
            weights=probabilities,
            k=1
        )[0]

        selected, cost = choices[selected_index]

        # Update resource availability
        if selected == "WAIT":

            allocation[pid] = "Waiting"

        else:

            b, e = selected

            remaining_beds[b] -= 1
            remaining_equipment[e] -= 1
            available_staff -= p["staff"]

            allocation[pid] = (b, e)

        total_cost += cost

    return allocation, total_cost


# -------------------------------
# 4. ACO OPTIMIZATION
# -------------------------------

best_allocation = None
best_cost = float("inf")

for iteration in range(iterations):

    iteration_best = None
    iteration_cost = float("inf")

    for ant in range(ants):

        allocation, cost = construct_solution()

        if cost < iteration_cost:
            iteration_cost = cost
            iteration_best = allocation

        if cost < best_cost:
            best_cost = cost
            best_allocation = allocation.copy()

    # Evaporation
    for key in pheromone:
        pheromone[key] *= (1 - evaporation)

    # Deposit pheromone using best solution
    for pid, resource in iteration_best.items():

        if resource == "Waiting":
            key = (pid, "WAIT")
        else:
            key = (pid, resource)

        pheromone[key] += Q / (1 + iteration_cost)

    print(
        "Iteration:", iteration + 1,
        "| Best Cost:", best_cost
    )


# -------------------------------
# 5. DISPLAY BEST ALLOCATION
# -------------------------------

print("\n========== HOSPITAL RESOURCE ALLOCATION ==========")

allocated = 0
waiting = 0

for p in patients:

    pid = p["id"]
    result = best_allocation[pid]

    print("\nPatient:", pid)
    print("Priority:", p["priority"])

    if result == "Waiting":

        print("Status: WAITING")
        waiting += 1

    else:

        print("Status: ALLOCATED")
        print("Bed:", result[0])
        print("Equipment:", result[1])
        print("Staff required:", p["staff"])

        allocated += 1

print("\n========== FINAL RESULT ==========")
print("Allocated Patients:", allocated)
print("Waiting Patients:", waiting)
print("Minimum Allocation Cost:", best_cost)


Iteration: 1 | Best Cost: 793
Iteration: 2 | Best Cost: 793
Iteration: 3 | Best Cost: 793
Iteration: 4 | Best Cost: 793
Iteration: 5 | Best Cost: 793
Iteration: 6 | Best Cost: 793
Iteration: 7 | Best Cost: 793
Iteration: 8 | Best Cost: 793
Iteration: 9 | Best Cost: 793
Iteration: 10 | Best Cost: 793
Iteration: 11 | Best Cost: 793
Iteration: 12 | Best Cost: 793
Iteration: 13 | Best Cost: 793
Iteration: 14 | Best Cost: 793
Iteration: 15 | Best Cost: 793
Iteration: 16 | Best Cost: 793
Iteration: 17 | Best Cost: 793
Iteration: 18 | Best Cost: 793
Iteration: 19 | Best Cost: 793
Iteration: 20 | Best Cost: 793
Iteration: 21 | Best Cost: 793
Iteration: 22 | Best Cost: 793
Iteration: 23 | Best Cost: 793
Iteration: 24 | Best Cost: 793
Iteration: 25 | Best Cost: 793
Iteration: 26 | Best Cost: 793
Iteration: 27 | Best Cost: 793
Iteration: 28 | Best Cost: 793
Iteration: 29 | Best Cost: 793
Iteration: 30 | Best Cost: 793
Iteration: 31 | Best Cost: 793
Iteration: 32 | Best Cost: 793
Iteration: 33 | B